# JAX tutorial
このノートブックは、Jaxを用いた実装をテストするものです。そもそもTPUを使えているかを確認する方法、使用可能なメモリ量、そしてPytorchと比べてどの程度差があるのかを確認します。

## TPU使用の確認
　以下のコードで、使用されているデバイスを確認できます。

In [1]:
import jax

# Check if TPU is available (JAX)
print("JAX version:", jax.__version__)
print("\nAvailable devices:")
print(jax.devices())
print("\nNumber of devices:", len(jax.devices()))
print("\nDevice type:", jax.devices()[0].device_kind if len(jax.devices()) > 0 else "No devices")

# Check if TPU is being used
is_tpu = any('tpu' in str(device).lower() for device in jax.devices())
print(f"\nTPU is available: {is_tpu}")

if is_tpu:
    print("✓ TPU is ready to use!")
else:
    print("⚠ TPU is not available. Using CPU/GPU instead.")


/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


JAX version: 0.7.2

Available devices:
[TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]

Number of devices: 1

Device type: TPU v5 lite

TPU is available: True
✓ TPU is ready to use!


## メモリ量の確認

以下のコードで使用可能なメモリ量を確認できます。無料版だと16GB程度が利用可能であるようです。

In [19]:
import jax

# List all devices JAX can see
available_devices = jax.devices()
print(f"Available devices: {available_devices}")

# Helper function to format bytes
def format_bytes(bytes_val):
    """Convert bytes to human-readable format"""
    for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
        if bytes_val < 1024.0:
            return f"{bytes_val:.2f} {unit}"
        bytes_val /= 1024.0
    return f"{bytes_val:.2f} PB"

# TPUで扱える最大メモリを表示
for device in available_devices:
    # Check if device is TPU
    if "tpu" in str(device).lower():
        print(f"\nTPU device: {device}")
        
        # Method 1: Try memory_size attribute (not available for TPU)
        memory = getattr(device, 'memory_size', None)
        if memory is not None:
            mem_gb = memory / (1024 ** 3)
            print(f"  最大メモリ (memory_size): {mem_gb:.2f} GB")
        else:
            print(f"  memory_size属性: 利用不可（TPUでは直接取得できません）")
        
        # Method 2: Use memory_stats() method (available for TPU)
        try:
            stats = device.memory_stats()
            if stats:
                print(f"  メモリ統計情報 (memory_stats):")
                print(f"    - 現在使用中: {format_bytes(stats.get('bytes_in_use', 0))}")
                print(f"    - ピーク使用量: {format_bytes(stats.get('peak_bytes_in_use', 0))}")
                print(f"    - メモリ上限: {format_bytes(stats.get('bytes_limit', 0))}")
                print(f"    - 利用可能な最大ブロック: {format_bytes(stats.get('largest_free_block_bytes', 0))}")
        except Exception as e:
            print(f"  memory_stats()も利用不可: {e}")
        
        print(f"\n  ✓ TPUは動作しています！計算結果から確認できます（Cell 6参照）")



Available devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]

TPU device: TPU_0(process=0,(0,0,0,0))
  memory_size属性: 利用不可（TPUでは直接取得できません）
  メモリ統計情報 (memory_stats):
    - 現在使用中: 739.25 MB
    - ピーク使用量: 1.20 GB
    - メモリ上限: 15.75 GB
    - 利用可能な最大ブロック: 14.54 GB

  ✓ TPUは動作しています！計算結果から確認できます（Cell 6参照）


## 大規模行列の計算例
TPUを使った計算の速度を表示します。CPUを使ったNumPyと比べ、JAXでは非常に早くなっていることが確認できます。

In [20]:
# TPUで顕著な計算高速化が得られる例として、大規模な行列乗算（matrix multiplication）をJAXのJIT (Just-In-Time) コンパイルで実行してみます。

from functools import partial
import numpy as np
import jax.numpy as jnp
from jax import jit
import time

# 行列サイズを大きくするとTPUによる高速化効果が顕著になります
SIZE = 8000

@partial(jit, static_argnums=())
def tpu_matmul(a, b):
    return jnp.dot(a, b)

def cpu_matmul(a, b):
    return np.dot(a, b)

# ランダム行列を作成
np.random.seed(0)
a_np = np.random.randn(SIZE, SIZE).astype(np.float32)
b_np = np.random.randn(SIZE, SIZE).astype(np.float32)
a_jnp = jnp.array(a_np)
b_jnp = jnp.array(b_np)

import jax

print(f"\n==== JAXが利用しているデバイス一覧 ====")
for i, device in enumerate(jax.devices()):
    print(f"Device {i}: {device}")

# 今回、デフォルトデバイス（計算に使うデバイス）を表示
print(f"JAXのデフォルトデバイス: {jax.devices()[0]}")

# NumPy (CPU) の計算時間
print(f"\n==== 大規模行列乗算（サイズ: {SIZE} x {SIZE}）====")
start = time.time()
cpu_matmul(a_np, b_np)
cpu_time = time.time() - start
print(f"NumPy (CPU) 実行時間: {cpu_time:.3f} 秒")

# JAX (TPU/GPU) の計算時間（JITコンパイル: 最初はウォームアップ、2回目が本番）
_ = tpu_matmul(a_jnp, b_jnp).block_until_ready()  # JITコンパイル
start = time.time()
_ = tpu_matmul(a_jnp, b_jnp).block_until_ready()
jax_time = time.time() - start
print(f"JAX ({jax.devices()[0].device_kind}, JIT) 実行時間: {jax_time:.3f} 秒")

speedup = cpu_time / jax_time if jax_time > 0 else np.nan
print(f"{jax.devices()[0].device_kind} による高速化倍率: {speedup:.2f}倍")



==== JAXが利用しているデバイス一覧 ====
Device 0: TPU_0(process=0,(0,0,0,0))
JAXのデフォルトデバイス: TPU_0(process=0,(0,0,0,0))

==== 大規模行列乗算（サイズ: 8000 x 8000）====
NumPy (CPU) 実行時間: 6.234 秒
JAX (TPU v5 lite, JIT) 実行時間: 0.007 秒
TPU v5 lite による高速化倍率: 940.52倍


## 自動微分

JAXには「自動微分」と呼ばれる機能があり、数式で定義した関数の導関数（微分関数）を自動的に生成できます。例えば次のような三次関数

$$
f(x) = x^3 + 2x^2 - 3x + 1
$$

の導関数

$$
\frac{df}{dx} = 3x^2 + 4x - 3
$$

を、JAXを使ってコードとして自動生成することができます。  
例えば、
$$
f(1)=1 \\
f'(1)=4
$$
になるか確認してみましょう。

In [25]:
f = lambda x: x**3 + 2*x**2 - 3*x + 1

dfdx = jax.grad(f)
x=1.
print(f"f(x) = {f(x)}, df/dx = {dfdx(x)}")






f(x) = 1.0, df/dx = 4.0


ではここで、関数が高次元になったらどうでしょうか？
例えば、2変数関数の場合を考えてみましょう。

$$
g(\mathbf{x}) = 3x_1^2 + 2x_1x_2 + x_2^2
$$

JAXで偏微分を自動計算できます。例えば $\mathbf{x} = [1.0, 2.0]$ のとき、

$$
\frac{\partial g}{\partial x_1} = 6x_1 + 2x_2\\
\frac{\partial g}{\partial x_2} = 2x_1 + 2x_2
$$

となります。

これをJAXで計算してみましょう。例えば、$\mathbf{x} = [1.0, 2.0]$ のとき、g(x) の値や偏微分の値がどうなるかも確認してみます。
このとき、$g([1.0, 2.0]) = 11.0$、$\frac{\partial g}{\partial x_1} = 10.0$、$\frac{\partial g}{\partial x_2} = 6.0$ となります。

In [26]:
import jax
import jax.numpy as jnp

def g(x):
    return 3 * x[0]**2 + 2 * x[0]*x[1] + x[1]**2

grad_g = jax.grad(g)
x = jnp.array([1.0, 2.0])
print("g(x) =", g(x))
print("∂g/∂x1 =", grad_g(x)[0])
print("∂g/∂x2 =", grad_g(x)[1])


g(x) = 11.0
∂g/∂x1 = 10.0
∂g/∂x2 = 6.0
